In [1]:
import os
import sys
import math
import copy 
import torch 
import random
DEBUG = False
import numpy as np
import torch_sparse
import pickle as pkl
import pandas as pd 
from tqdm import tqdm
from time import time
import networkx as nx
from dgl import DGLGraph
from scipy import linalg
from pathlib import Path
from torch import Tensor
import scipy.sparse as sp
from random import randint
from dgl import transforms
device = torch.device("cpu")
from scipy import sparse, stats
from scipy.sparse import csgraph
from scipy.sparse import csr_matrix
from dgl import from_networkx, DGLGraph
from dgl.data import citation_graph as citegrh
from torch_geometric.typing import SparseTensor
from torch_geometric.utils import to_undirected
from torch_geometric.utils import remove_self_loops
from sklearn.metrics.pairwise import cosine_similarity
from ipynb.fs.full.Dataset import get_data_from_dataset
from typing import Callable, List, NamedTuple, Optional, Tuple, Union
from torch_geometric.utils import add_self_loops,add_remaining_self_loops
from ipynb.fs.full.SpectralSparsifier import EffectiveResistance, LocalEffectiveResistance, get_sparse_adj_matrix

/home/muftiqur/Graph Research/Graph-Sparsification/GraphSparsification/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directory:  ./Dataset/
Result directory: ./Dataset/RESULTS/

Dataset: Amherst41(1):
Number of graphs: 1
Number of features: 1193
Number of classes: 2

Data(x=[2235, 1193], edge_index=[2, 181908], y=[2235], train_mask=[2235], val_mask=[2235], test_mask=[2235])
Number of nodes: 2235
Number of edges: 181908
Average node degree: 81.39
Number of training nodes: 1341
Training node label rate: 0.60
Has isolated nodes: False
Has self-loops: False
Is undirected: True
Shifting label to non-negative
Cpu count:  12


In [2]:
from torch_geometric.datasets import WebKB
from torch_geometric.datasets import CoraFull
from torch_geometric.datasets import Planetoid
from torch_geometric.datasets import Reddit, Reddit2
from torch_geometric.transforms import NormalizeFeatures

if os.uname()[1].find('gilbreth')==0: ##if not darwin(mac/locallaptop)
    DIR='/scratch/gilbreth/das90/Dataset/'
elif os.uname()[1].find('unimodular')==0:
    DIR='/scratch2/das90/Dataset/'
elif os.uname()[1].find('Siddharthas')==0:
    DIR='/Users/siddharthashankardas/Purdue/Dataset/'  
else:
    DIR='./Dataset/' 
Path(DIR).mkdir(parents=True, exist_ok=True)
RESULTS_DIR=DIR+'RESULTS/'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print("Data directory: ", DIR)
print("Result directory:", RESULTS_DIR)


Data directory:  ./Dataset/
Result directory: ./Dataset/RESULTS/


In [3]:
def get_data(DATASET_NAME):
        
    #DATASET_NAME='Cora' #"Cora", "CiteSeer", "PubMed"
    if DATASET_NAME in ["Cora", "CiteSeer", "PubMed"]:
        dataset = Planetoid(root=DIR+'Planetoid', name=DATASET_NAME, transform=NormalizeFeatures())

    elif DATASET_NAME == "Reddit2":
        dataset = Reddit2(root=DIR+'Reddit2', transform=NormalizeFeatures())

    elif DATASET_NAME == "Reddit":
        dataset = Reddit(root=DIR+'Reddit', transform=NormalizeFeatures())
    elif DATASET_NAME == "Texas":
        dataset = WebKB(root=DIR+'Texas', name="Texas",transform=NormalizeFeatures())
    elif DATASET_NAME == "Wisconsin":
            dataset = WebKB(root=DIR+'Wisconsin', name="Wisconsin",transform=NormalizeFeatures())
    elif DATASET_NAME == "Cornell":
         dataset = WebKB(root=DIR+'Cornell', name="Cornell",transform=NormalizeFeatures())
    else:    
        raise Exception('dataset not found')

    print()
    print(f'Dataset: {dataset}:')
    print('======================')
    print(f'Number of graphs: {len(dataset)}')
    print(f'Number of features: {dataset.num_features}')
    print(f'Number of classes: {dataset.num_classes}')

    data = dataset[0]  # Get the first graph object.

    print()
    print(data)
    print('===========================================================================================================')

    # Gather some statistics about the graph.
    print(f'Number of nodes: {data.num_nodes}')
    print(f'Number of edges: {data.num_edges}')
    print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
    print(f'Number of training nodes: {data.train_mask.sum()}')
    print(f'Training node label rate: {int(data.train_mask.sum()) / data.num_nodes:.2f}')
    print(f'Has isolated nodes: {data.has_isolated_nodes()}')
    print(f'Has self-loops: {data.has_self_loops()}')
    print(f'Is undirected: {data.is_undirected()}')
    
    return data, dataset

#data,dataset = get_data('Cora')

In [4]:
# #cosine_similarity_dictionary
def ER_(u,v, L_inv,N):
    x_u = np.zeros((N,))
    x_v = np.zeros((N,)) 
    x_u[u] = 1
    x_v[v] = 1
    d_uv=x_u-x_v
    R_uv=d_uv.dot(L_inv.dot(d_uv)) ## (x_u-x_v)^T*L'*(x_u-x_v)
    return R_uv

def compute_ER_(Adj, L_inv):
    start_nodes, end_nodes, weights = sparse.find(Adj)

    n = np.shape(Adj)[0]
    Reff = sparse.lil_matrix((n,n))
    for orig, end in zip(start_nodes, end_nodes):
        Reff[orig,end] = ER_(orig, end, L_inv, n)  
    return Reff

def EffectiveResistance(Adj):
    #Adj = nx.adjacency_matrix(G)
    L, D  = csgraph.laplacian(Adj, normed=False, return_diag=True)
    #print(np.allclose(L.todense(), np.diag(D)-Adj)) #verify L=D-A 
    L_inv = linalg.pinv(L.todense())
    Reff=compute_ER_(Adj, L_inv)
    return Reff

def ERprob(Adjacency_matrix, adj_t, compute='exact'):
    DEBUG = True 
    rowptr_, col_,_ = adj_t.csr()
    start_nodes, end_nodes, weights = sparse.find(Adjacency_matrix)
    if DEBUG: print("Computing edge resistances: ... ",compute)

    if compute=='exact':
        Re=EffectiveResistance(Adjacency_matrix)
        Re = np.maximum(0, Re[start_nodes, end_nodes].toarray())
    else:
        print("Computing Local Effective Resistance")
        Re=LocalEffectiveResistance(adj_t, method=None, eps=0.4, lmbda=0.1)
  
    if DEBUG: print("Finished computing resistances:")
    N=Adjacency_matrix.shape[0]
    # Calculate the new weights.
    weights = np.maximum(0, weights)
    Pe = weights * Re
    Pe_list = Pe + 0.0001
    for i in range(len(rowptr_) - 1):
        rowstart = rowptr_[i]
        rowend = rowptr_[i + 1]
        Pe_values = Pe_list[rowstart:rowend]
        #Normalizing Pe values According to their Neighbors
        Pe_sums = np.sum(Pe_values)  
        if Pe_sums == 0:
            Pe_values = 0
        else: 
            Pe_values/= Pe_sums
        #Pe_values /= Pe_sums 
        Pe_list[rowstart:rowend] = Pe_values
    ERW = sparse.csc_matrix((Pe_list, (start_nodes, end_nodes)),shape=(N, N))    
    ERW = ERW + ERW.T #making symmetric
    return Pe_list, ERW, Adjacency_matrix

def calculate_selection_probability_from_ER(adj_t,effective_resistance_algorithm='exact'):
    rowptr, col , _ = adj_t.csr()
    Adj = np.zeros((len(rowptr) - 1, len(rowptr) - 1))
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            weight = 1
            Adj[source_vertex, target_vertex] = weight
    sparse_adj = csr_matrix(Adj)
    Pe, ERW, Adj = ERprob(sparse_adj, adj_t, compute=effective_resistance_algorithm)
    return Pe

In [5]:
def calculate_cosine_similarity(key_vector, value_vectors):
    similarities = cosine_similarity([key_vector], value_vectors)
    return similarities[0]

def normalize_weights(weights):
    total_sum = sum(weights)
    normalized_weights = [w / total_sum for w in weights]
    return normalized_weights

def Custom_CosineSimilarity_Sampling(x, edge_index):
    cosine_similarity_dict = {}
    for node in range(x.size(0)):
        subset_index = torch.where(edge_index[1] == node)
        indices_list = subset_index[0].tolist()
        subset_nodes = edge_index[0, indices_list]
        neighbor_nodes = subset_nodes.tolist()
        key_vector = x[node].cpu().numpy()
        value_vectors = x[neighbor_nodes].cpu().numpy()
        similarities = calculate_cosine_similarity(key_vector, value_vectors)
        similarities = similarities + 0.000001
        normalized_weights = normalize_weights(similarities)
        cosine_similarity_dict[node] = {'neighbors': neighbor_nodes, 'similarities': normalized_weights}
    return cosine_similarity_dict

def calculate_weights(adj_t, cosine_similarity_dictionary):
    rowptr,col,_ = adj_t.csr()
    weights = []
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            target_index = cosine_similarity_dictionary[source_vertex]['neighbors'].index(target_vertex)
            edge_weight = cosine_similarity_dictionary[source_vertex]['similarities'][target_index]
            weights.append(edge_weight)
    return weights

In [8]:
def get_Adj_Matrix(edge_index: Union[Tensor, SparseTensor],
                 sizes: List[int], node_idx: Optional[Tensor] = None,
                 num_nodes: Optional[int] = None, return_e_id: bool = True,
                 transform: Callable = None,**kwargs):
    edge_index = edge_index.to('cpu')
    kwargs.pop('dataset', None)
    kwargs.pop('collate_fn', None)
    drop_last = kwargs.pop('drop_last', False)

    is_sparse_tensor = isinstance(edge_index, SparseTensor)
    __val__ = None
    
    if not is_sparse_tensor:
        if (num_nodes is None and node_idx is not None
                and node_idx.dtype == torch.bool):
            num_nodes = node_idx.size(0)
        if (num_nodes is None and node_idx is not None
                and node_idx.dtype == torch.long):
            num_nodes = max(int(edge_index.max()), int(node_idx.max())) + 1
        if num_nodes is None:
            num_nodes = int(edge_index.max()) + 1

        value = torch.arange(edge_index.size(1)) if return_e_id else None
        adj_t = SparseTensor(row=edge_index[0], col=edge_index[1],
                                    value=value,
                                    sparse_sizes=(num_nodes, num_nodes)).t()
    else:
        adj_t = edge_index
        if return_e_id:
            __val__ = adj_t.storage.value()
            value = torch.arange(adj_t.nnz())
            adj_t = adj_t.set_value(value, layout='coo')
        adj_t = adj_t

    adj_t.storage.rowptr()

    return adj_t

# Getting Cosine Weight List 
def precomputing_weight_calculation(x,edge_index: Union[Tensor, SparseTensor],
                 sizes: List[int], node_idx: Optional[Tensor] = None,
                 num_nodes: Optional[int] = None, return_e_id: bool = True,
                 transform: Callable = None,**kwargs):
        
        adj_t = get_Adj_Matrix(edge_index, node_idx = node_idx, sizes=sizes,num_nodes=num_nodes,return_e_id=return_e_id,transform=transform)
        cosine_similarity_dictionary = Custom_CosineSimilarity_Sampling(x,edge_index)
        weight = calculate_weights(adj_t, cosine_similarity_dictionary)
        return torch.tensor(weight)

In [9]:
# from torch_geometric.utils import add_remaining_self_loops, to_undirected
#data, dataset = get_data("Roman-empire")
#data.edge_index = add_remaining_self_loops(data.edge_index)[0]
#data.edge_index = to_undirected(data.edge_index)
#adj_t = precomputing_weight_calculation(data.x,data.edge_index,[4,4]) 
# data, dataset = get_data("Cornell")
# data.edge_index = add_remaining_self_loops(data.edge_index)[0]
# weight_list  = precomputing_weight_calculation(data.x,data.edge_index,[4,4])

In [10]:
def manage_dataset_directory(dataset_name,data, recompute=False):    
    base_dir = os.path.join(os.getcwd(),'Weight_List')
    #folder_name = os.path.join(base_dir,dataset_name) 
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    
    weight_list_file = dataset_name + '_weight_list.pt'
    weight_list_path = os.path.join(base_dir,weight_list_file)
    
    if recompute:
        print("Going to calculating Precomputing Edge Weights")
        #weight_list = precomputing_weight_calculation(data.x,data.edge_index,sizes=[8,4])  
        #torch.save(weight_list, weight_list_path)
    else:
        if os.path.exists(weight_list_path):
            weight_list = torch.load(weight_list_path)
        else:
            raise FileNotFoundError(f"The weight list file {weight_list_path} does not exist. Set recompute=True to create it.")
    return weight_list

In [11]:
# from torch_geometric.utils import remove_self_loops
# data,dataset = get_data_from_dataset('Roman-empire')
# edge_index = remove_self_loops(data.edge_index)[0]
# cosine_weight_list = precomputing_weight_calculation(data.x,edge_index,[2,2]) + 0.001
# ER_weight_list = manage_dataset_directory( dataset_name=dataset.name,data=data) + 0.001
# weight_list = ER_weight_list * cosine_weight_list
# weight_list

In [12]:
def weighted_sample_adj_cpu_(rowptr, col, idx, num_neighbors,weight, replace):
    assert rowptr.device.type == 'cpu'
    assert col.device.type == 'cpu'
    assert idx.device.type == 'cpu'
    assert idx.dim() == 1
    rowptr_data = rowptr.numpy()
    col_data = col.numpy()
    idx_data = idx.numpy()

    out_rowptr = torch.empty(idx.numel() + 1, dtype=torch.int64)
    out_rowptr_data = out_rowptr.numpy()
    out_rowptr_data[0] = 0

    cols = []
    n_ids = []
    n_id_map = {}
    for n in range(idx.numel()):
        i = idx_data[n]
        cols.append([])
        n_id_map[i] = n
        n_ids.append(i)
    if num_neighbors < 0:
        for i in range(idx.numel()):
            n = idx_data[i]
            row_start, row_end = rowptr_data[n], rowptr_data[n + 1]
            row_count = row_end - row_start

            for j in range(row_count):
                e = row_start + j
                c = col_data[e]

                if c not in n_id_map:
                    n_id_map[c] = len(n_ids)
                    n_ids.append(c)
                cols[i].append((n_id_map[c], e))
            out_rowptr_data[i + 1] = out_rowptr_data[i] + len(cols[i])
    
    elif replace:
        for i in range(idx.numel()):
            n = idx_data[i]
            row_start, row_end = rowptr_data[n], rowptr_data[n + 1]
            row_count = row_end - row_start
            if row_count > 0:
                window_weights = weight[row_start:row_end]
                # if window_weights.sum() <= 0:
                #     window_weights = torch.clamp(window_weights, min=0.0000001)
                neighbor_indices = random.choices(range(row_count), weights=window_weights, k=num_neighbors)
                for neighbor_index in neighbor_indices:
                    e = row_start + neighbor_index
                    c = col_data[e]
                    
                    if c not in n_id_map:
                        n_id_map[c] = len(n_ids)
                        n_ids.append(c)
                    cols[i].append((n_id_map[c], e))
                out_rowptr_data[i + 1] = out_rowptr_data[i] + len(cols[i])
    else:
        for i in range(idx.numel()):
            n = idx_data[i]
            row_start, row_end = rowptr_data[n], rowptr_data[n + 1]
            row_count = row_end - row_start
            perm = set()

            if row_count <= num_neighbors:
                for j in range(row_count):
                    perm.add(j)
            else:
                indx_list = list(range(0,row_count))
                window_weights = weight[row_start:row_end]    
                while len(perm) < num_neighbors:
                    neighbor = random.choices(indx_list, weights=window_weights, k=1)[0]
                    if neighbor not in perm and col_data[row_start+neighbor]!=n:
                        perm.add(neighbor)
                        
            for p in perm:
                e = row_start + p
                c = col_data[e]
                if c not in n_id_map:
                    n_id_map[c] = len(n_ids)
                    n_ids.append(c)
                cols[i].append((n_id_map[c], e))

            out_rowptr_data[i + 1] = out_rowptr_data[i] + len(cols[i])
    
    N = len(n_ids)
    out_n_id = torch.tensor(n_ids, dtype=torch.int64)

    E = out_rowptr_data[-1]
    out_col = torch.empty((E,), dtype=torch.int64)
    out_e_id = torch.empty((E,), dtype=torch.int64)
    out_col_data = out_col.numpy()
    out_e_id_data = out_e_id.numpy()
    i = 0
    for col_vec in cols:
        col_vec.sort(key=lambda x: x[0])
        for value in col_vec:
            out_col_data[i] = value[0]
            out_e_id_data[i] = value[1]
            i += 1
    return out_rowptr, out_col, out_n_id, out_e_id

def sample_adj_(src: SparseTensor, subset: torch.Tensor, num_neighbors: int,
               weight: List[float], replace: bool = False) -> Tuple[SparseTensor, torch.Tensor]:
    rowptr, col, value = src.csr() #rowptr, col, n_id, e_id = torch.ops.torch_sparse.sample_adj(rowptr, col, subset, num_neighbors, replace)
    rowptr, col, n_id, e_id = weighted_sample_adj_cpu_(rowptr, col, subset, num_neighbors,weight, replace)
    if value is not None:
        value = value[e_id]
    out = SparseTensor(rowptr=rowptr, row=None, col=col, value=value,
                       sparse_sizes=(subset.size(0), n_id.size(0)),
                       is_sorted=True)
    
    return out, n_id

In [13]:
class EdgeIndex(NamedTuple):
    edge_index: Tensor
    e_id: Optional[Tensor]
    size: Tuple[int, int]

    def to(self, *args, **kwargs):
        edge_index = self.edge_index.to(*args, **kwargs)
        e_id = self.e_id.to(*args, **kwargs) if self.e_id is not None else None
        return EdgeIndex(edge_index, e_id, self.size)

class Adj(NamedTuple):
    adj_t: SparseTensor
    e_id: Optional[Tensor]
    size: Tuple[int, int]

    def to(self, *args, **kwargs):
        adj_t = self.adj_t.to(*args, **kwargs)
        e_id = self.e_id.to(*args, **kwargs) if self.e_id is not None else None
        return Adj(adj_t, e_id, self.size)

#Taken from https://github.com/rusty1s/pytorch_sparse/blob/master/torch_sparse/sample.py
# Convert this cpp into Python https://github.com/rusty1s/pytorch_sparse/blob/master/csrc/cpu/sample_cpu.cpp
# Weighted Sampler https://github.com/siddhartha047/Graph-Sparsification/blob/main/Submodular/AGSNodeSampler.ipynb
# rowptr, col, n_id, e_id = torch.ops.torch_sparse.sample_adj(rowptr, col, subset, num_neighbors, replace)
class NeighborSampler_(torch.utils.data.DataLoader):
    def __init__(self, edge_index: Union[Tensor, SparseTensor],
                 sizes: List[int], node_idx: Optional[Tensor] = None,
                 num_nodes: Optional[int] = None, return_e_id: bool = True,
                 transform: Callable = None,weight_list=None, **kwargs):

        edge_index = edge_index.to('cpu')
        kwargs.pop('dataset', None)
        kwargs.pop('collate_fn', None)
        self.drop_last = kwargs.pop('drop_last', False)
        self.edge_index = edge_index
        self.node_idx = node_idx
        self.num_nodes = num_nodes

        self.sizes = sizes
        self.return_e_id = return_e_id
        self.transform = transform
        self.is_sparse_tensor = isinstance(edge_index, SparseTensor)
        self.__val__ = None
        self.weight = weight_list

        if not self.is_sparse_tensor:
            if (num_nodes is None and node_idx is not None
                    and node_idx.dtype == torch.bool):
                num_nodes = node_idx.size(0)
            if (num_nodes is None and node_idx is not None
                    and node_idx.dtype == torch.long):
                num_nodes = max(int(edge_index.max()), int(node_idx.max())) + 1
            if num_nodes is None:
                num_nodes = int(edge_index.max()) + 1

            value = torch.arange(edge_index.size(1)) if return_e_id else None
            self.adj_t = SparseTensor(row=edge_index[0], col=edge_index[1],
                                      value=value,
                                      sparse_sizes=(num_nodes, num_nodes)).t()
        else:
            adj_t = edge_index
            if return_e_id:
                self.__val__ = adj_t.storage.value()
                value = torch.arange(adj_t.nnz())
                adj_t = adj_t.set_value(value, layout='coo')
            self.adj_t = adj_t

        self.adj_t.storage.rowptr()

        if node_idx is None:
            node_idx = torch.arange(self.adj_t.sparse_size(0))
        elif node_idx.dtype == torch.bool:
            node_idx = node_idx.nonzero(as_tuple=False).view(-1)

        super().__init__(
            node_idx.view(-1).tolist(), collate_fn=self.sample, **kwargs)
        
    def get_weight(self):
        return self.weight
        
    def get_adj(self):
        return self.adj_t
        
    def sample(self, batch):
        if not isinstance(batch, Tensor):
            batch = torch.tensor(batch)
        batch_size: int = len(batch)
        adjs = []
        n_id = batch
        for size in self.sizes:
            #adj_t,n_id = self.adj_t.sample_adj(n_id, size, replace=False)
            adj_t,n_id = sample_adj_(self.adj_t, n_id, size, self.weight, replace=True)
            #adj_t,n_id = sample_adj(self.adj_t,n_id, size, replace=False)
            #subgraph_all_nodes = get_node_ids_(cosine_similarity_dictionary, n_id,size)
            #adj_t = create_subgraph_adjacency_(n_id,subgraph_all_nodes,self.adj_t)
            #n_id = subgraph_all_nodes
            e_id = adj_t.storage.value()
            size = adj_t.sparse_sizes()[::-1]

            if self.__val__ is not None:
                adj_t.set_value_(self.__val__[e_id], layout='coo')

            if self.is_sparse_tensor:
                adjs.append(Adj(adj_t, e_id, size))
            else:
                row, col, _ = adj_t.coo()
                edge_index = torch.stack([col, row], dim=0)
                adjs.append(EdgeIndex(edge_index, e_id, size))
        adjs = adjs[0] if len(adjs) == 1 else adjs[::-1]
        out = (batch_size, n_id, adjs)
        out = self.transform(*out) if self.transform is not None else out
        return out
    
    def __repr__(self) -> str:
        return f'{self.__class__.__name__}(sizes={self.sizes})'

In [14]:
def get_sparse_adj_matrix(adj_t):
    rowptr, col, _ = adj_t.csr()
    Adj = np.zeros((len(rowptr) - 1, len(rowptr) - 1))
    j = 0
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            weight = 1
            Adj[source_vertex, target_vertex] = weight
            j += 1  
    sparse_adj = csr_matrix(Adj)
    return sparse_adj 

In [24]:
# data, dataset = get_data_from_dataset('Cora')
# edge_index = remove_self_loops(data.edge_index)[0]
# edge_index = to_undirected(data.edge_index)
# cos_weight = precomputing_weight_calculation(data.x,edge_index,[4,4])

In [23]:
# train_idx = data.train_mask.nonzero(as_tuple=False).view(-1)
# train_loader = NeighborSampler_(edge_index, node_idx=train_idx,
#                                    sizes=[4,4], batch_size=64,
#                                    shuffle=False, num_workers=1,weight_list=cos_weight) 
# for batch_size, n_id, adjs in train_loader:
#     print(n_id)
#     print("HELLO WORLD")

In [14]:
# data, dataset = get_data_from_dataset('Cora')
# ER_weight = manage_dataset_directory(dataset.name,data)

In [15]:
# edge_index = data.edge_index 
# edge_index = remove_self_loops(data.edge_index)[0]
# train_idx = data.train_mask.nonzero(as_tuple=False).view(-1)
# train_loader = NeighborSampler_(edge_index, node_idx=train_idx,
#                                    sizes=[4,4], batch_size=4,
#                                    shuffle=False, num_workers=4) 
# for batch_size, n_id, adjs in train_loader:
#     print(n_id)
#     print("HELLO WORLD")

In [16]:
# data, dataset = get_data_from_dataset('Physics')
# ER_weight = manage_dataset_directory(dataset.name,data)
# ER_weight, len(ER_weight)

In [17]:
# edge_index = data.edge_index 
# edge_index = remove_self_loops(data.edge_index)[0]
# train_idx = data.train_mask.nonzero(as_tuple=False).view(-1)
# train_loader = NeighborSampler_(edge_index, node_idx=train_idx,
#                                    sizes=[4,4], batch_size=64,
#                                    shuffle=False, num_workers=1,weight_list=ER_weight) 
# for batch_size, n_id, adjs in train_loader:
#     print("HELLO WORLD")

## Calculating effective resistance

In [18]:
from dgl.data import citation_graph as citegrh
from dgl.data import reddit
from dgl.data import gnn_benchmark as gnnbnch
from dgl.data import ActorDataset
from dgl.data import CornellDataset
from dgl.data import RomanEmpireDataset
from dgl.data import ChameleonDataset
from dgl.data import AmazonRatingsDataset
from dgl.data import WisconsinDataset
from dgl.data import TexasDataset
from dgl.data import AmazonCoBuyPhotoDataset
from dgl.data import MinesweeperDataset

from scipy import sparse
from scipy.sparse import csr_matrix
import numpy as np
import torch
import pandas as pd
from dgl import DGLGraph
import networkx as nx
import pandas as pd
from scipy import sparse, stats
from numpy import inf
import random
import dgl
from dgl import to_networkx, DGLGraph
from dgl import remove_self_loop
import scipy

In [19]:
from torch_geometric.utils import remove_self_loops

def generate_L(Ag, N):
    v = np.ones(N)
    Dv = Ag.dot(v)
    Dg = csr_matrix((Dv, (np.arange(N), np.arange(N))), shape=(N, N))
    Lg = Dg - Ag
    print("Finished building the Laplacian")
    return Lg

def compute_reff(W, V):
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(W))
    n = np.shape(W)[0]
    Reff = sparse.lil_matrix((n, n))
    for orig, end in zip(start_nodes, end_nodes):
        Reff[orig, end] = np.linalg.norm(V[orig, :] - V[end, :]) ** 2
    return Reff

def graph_sparsify(Lg, epsilon,filename, choice=1):
    # filenames = ['V_Actor.csv']
    # filename = filenames[choice - 1]
    print("Computing resistances for ", filename)
    N = np.size(Lg, 0)
    Dv = Lg.diagonal()
    Dg = csr_matrix((Dv, (np.arange(N), np.arange(N))), shape=(N, N))
    W = Dg - Lg
    read_V = [1, 3, 4, 5, 8, 9, 10, 11, 12, 13]
    if choice in read_V:
        print("Reading V matrix:..")
        V_frame = pd.read_csv(filename, header=None)
        V = V_frame.to_numpy()
        print("Computing edge resistances:... ")

        resistance_distances = compute_reff(W, V)
        print("Finished loading resistances:")
    else:
        resistance_distances = np.loadtxt(filename)
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(W))
    weights = np.maximum(0, weights)
    if choice in read_V:
        Re = np.maximum(0, resistance_distances[start_nodes, end_nodes].toarray())
        ER_weight_matrix = np.zeros((18,18))
        for i in range(len(start_nodes)):
            ER_weight_matrix[end_nodes[i],start_nodes[i]] = Re[0][i]
            ER_weight_matrix[start_nodes[i],end_nodes[i]] = Re[0][i]
    else:
        Re = np.maximum(0, resistance_distances[start_nodes, end_nodes])
    Pe = weights * Re
    Pe = Pe / np.sum(Pe)
    Pe = np.squeeze(Pe)
    # Rudelson, 1996 Random Vectors in the Isotropic Position (too hard to figure out actual C0)
    C0 = 1 / 30. # Rudelson and Vershynin, 2007, Thm. 3.1
    C = 4 * C0
    q = round(N * np.log(N) * 9 * C ** 2 / (epsilon ** 2))
    #results = stats.rv_discrete(values=(np.arange(np.shape(Pe)[0]), Pe)).rvs(size=int(q))
    random.seed(42)
    results = np.random.choice(np.arange(np.shape(Pe)[0]), int(q), p=list(Pe))
    #spin_counts = stats.itemfreq(results).astype(int)
    spin_counts = np.unique(results, return_counts=True)
    per_spin_weights = weights / (q * Pe)
    per_spin_weights[per_spin_weights == inf] = 0
    counts = np.zeros(np.shape(weights)[0]) #counts[spin_counts[:, 0]] = spin_counts[:, 1]
    counts[spin_counts[0]] = spin_counts[1]
    new_weights = counts * per_spin_weights
    sparserW = sparse.csc_matrix((np.squeeze(new_weights), (start_nodes, end_nodes)),
                                 shape=(N, N))
    sparserW = sparserW + sparserW.T
    return sparserW, np.count_nonzero(new_weights),ER_weight_matrix

def save_weight_list(data, dataset_name, Wsp):
    edge_index = remove_self_loops(data.edge_index)[0]
    weight_list = np.zeros(len(edge_index[0]))
    w_matrix = Wsp.toarray()
    for ind,(u,v) in enumerate(zip(data.edge_index[0],data.edge_index[1])):
        weight_list[ind] = w_matrix[u][v]
    weight_tensor = torch.tensor(weight_list)
    torch.save(weight_tensor, dataset_name + '_weight_list.pt')

def adjacency_matrix(data, num_nodes, scipy_fmt='csr'):
    adj_matrix = torch.zeros((num_nodes, num_nodes), dtype=torch.float)
    edge_index = remove_self_loops(data.edge_index)[0]
    for i in range(edge_index.shape[1]):
        source = edge_index[0, i].item()
        target = edge_index[1, i].item()
        adj_matrix[source, target] = 1
        adj_matrix[target, source] = 1
    # Convert the PyTorch tensor to a NumPy array
    adj_matrix_np = adj_matrix.numpy()
    # Convert the NumPy array to a SciPy CSR matrix if requested
    if scipy_fmt == 'csr':
        adj_matrix_csr = csr_matrix(adj_matrix_np)
        return adj_matrix_csr
    else:
        raise ValueError(f"Unsupported format: {scipy_fmt}")

def generate_spare_graph(Ag, N, filename, epsilon=0.6):
    #Ag = g.adj_external(scipy_fmt = 'csr')
    Lg = generate_L(Ag, N)
    print("Sparsifying the graph: ")
    Wsp, Ne_sp,Re = graph_sparsify(Lg, epsilon, filename)
    #Re = graph_sparsify(Lg, epsilon, filename)
    print("Finished sparsifying the graph: ")
    g_sp = DGLGraph() 
    #g_sp.from_scipy_sparse_matrix(Wsp)
    g_sp = dgl.from_scipy(Wsp)
    #adj_matrix = g_sp.adjacency_matrix().to_dense()
    # add self loop
    #g.add_edges(g.nodes(), g.nodes())
    #g_sp.add_edges(g_sp.nodes(), g_sp.nodes())
    return g_sp, Ne_sp, Wsp,Re

In [25]:
def get_graph(dataset_name):
    if dataset_name == 'Actor':
        data = ActorDataset()
        g = data[0]
        g = g.remove_self_loop()
        N = g.number_of_nodes()
    elif dataset_name == 'roman_empire':
        data, dataset = get_data_from_dataset('Roman-empire')
        N = data.num_nodes
        g = adjacency_matrix(data,N)
    elif dataset_name =='Amazon-ratings':
        data = AmazonRatingsDataset()
        g = data[0]
        g= g.remove_self_loop()
        N = g.number_of_nodes()
    elif dataset_name=='Wisconsin':
        data = WisconsinDataset()
        g = data[0]
        g = g.remove_self_loop()
        N = g.number_of_nodes()
    elif dataset_name=='reed98':
        data,dataset = get_data_from_dataset('reed98')
        N = data.num_nodes
        g = adjacency_matrix(data,N)
    elif dataset_name == 'amherst41':
        data, dataset = get_data_from_dataset('amherst41')
        N = data.num_nodes 
        g = adjacency_matrix(data,N)
    elif dataset_name in ['photo','Photo']:
        data, dataset = get_data_from_dataset('Photo')
        N = data.num_nodes 
        g = adjacency_matrix(data,N)
    elif dataset_name =='dblp':
        data, dataset = get_data_from_dataset('dblp')
        N = data.num_nodes 
        g = adjacency_matrix(data,N)
    elif dataset_name == 'Minesweeper':
        data = MinesweeperDataset()
        g = data[0]
        g = g.remove_self_loop()
        N = g.number_of_nodes()
    return g,N

def save_npz_matrix(dataset_name,Ag=None):
    filename = dataset_name 
    if filename=='Actor':
        print("Saving NPZ of Actor Dataset")
        data, dataset = get_data_from_dataset('Actor')
        N = data.num_nodes
        print(N)
        Ag = adjacency_matrix(data,N)

    elif filename == 'Roman-empire':
        print("Saving NPZ of Roman-empire Dataset")
        data, dataset = get_data_from_dataset('Roman-empire')
        N = data.num_nodes
        print(N)
        Ag = adjacency_matrix(data,N)
    
    elif filename =='Chameleon':
        print("Saving NPZ of Chameleon Dataset")
        data = ChameleonDataset()
        g = data[0]
        g= g.remove_self_loop()
        N = g.number_of_nodes()
        Ag = g.adj_external(scipy_fmt = 'csr')

    elif filename =='Amazon-ratings':
        print("Saving NPZ of Amazon-ratings Dataset")
        data, dataset = get_data_from_dataset('Amazon-ratings')
        N = data.num_nodes
        Ag = adjacency_matrix(data,N)
        
    elif filename =='Wisconsin':
        print("Saving NPZ of Wisconsin Dataset")
        data = WisconsinDataset()
        g = data[0]
        g = g.remove_self_loop()
        N = g.number_of_nodes()
        Ag = g.adj_external(scipy_fmt = 'csr')
    elif filename =='dblp':
        print("Saving NPZ of dblp Dataset")
        data,dataset = get_data_from_dataset('dblp')
        Ag = adjacency_matrix(data,data.num_nodes) 

    elif filename =='Photo':
        print("Saving NPZ of Photo Dataset")
        data = AmazonCoBuyPhotoDataset()
        g = data[0]
        g = g.remove_self_loop()
        N = g.number_of_nodes()
        Ag = g.adj_external(scipy_fmt = 'csr')

    elif dataset_name == 'amherst41':
        data, dataset = get_data_from_dataset('amherst41')
        N = data.num_nodes
        print(N)
        Ag = adjacency_matrix(data,N)
        
    elif filename == 'Minesweeper':
        print('Saving Npz of Minesweeper Dataset')
        data = MinesweeperDataset()
        g = data[0]
        g = g.remove_self_loop()
        N = g.number_of_nodes()
        Ag = g.adj_external(scipy_fmt = 'csr')
        
    elif filename =='Texas':
        print("Saving Npz of Texas Dataset")
        data = TexasDataset()
        g = data[0]
        g = g.remove_self_loop()
        N = g.number_of_nodes()
        Ag = g.adj_external(scipy_fmt = 'csr')
    print("Saving data now.. ")
    sparse.save_npz(filename,Ag)

In [1]:
#save_npz_matrix(dataset_name='Amazon-ratings')

In [20]:
# data = MinesweeperDataset()
# g = data[0]
# g = g.remove_self_loop()
# N = g.number_of_nodes()
# filename = 'V_Minesweeper.csv'
# g_sp, Ne_sp, Wsp = generate_spare_graph(g, N, filename)

In [22]:
# g, N = get_graph('roman_empire')
# filename = 'V_Roman-empire.csv'
# g_sp, Ne_sp, Wsp = generate_spare_graph(g, N, filename)

In [22]:
# g, N = get_graph('Minesweeper')
# filename = 'V_Minesweeper.csv'
# g_sp, Ne_sp, Wsp = generate_spare_graph(g, N, filename)

In [23]:
# data, dataset = get_data_from_dataset('Minesweeper')

In [24]:
#save_weight_list(data, dataset_name=dataset.name, Wsp=Wsp)

In [25]:
# data, dataset = get_data_from_dataset('reed98')
# edge_index = remove_self_loops(data.edge_index)[0]
# ER_weight = manage_dataset_directory(dataset.name, data) + 0.001
# cosine_weight = precomputing_weight_calculation(data.x,edge_index,[8,4]) + 0.001
# weight_list = ER_weight*cosine_weight
# weight_list
# train_neighbors = [8,4]
# train_idx = data.train_mask.nonzero(as_tuple=False).view(-1)
# train_loader = NeighborSampler_(edge_index, node_idx=train_idx,
#                                    sizes=train_neighbors, batch_size=64,
#                                    shuffle=True, num_workers=1,weight_list=weight_list) 
# for batch_size, n_id, adjs in train_loader:
#     print("HELLO")

In [26]:
# data,dataset = get_data_from_dataset('amherst41')
# num_nodes = data.num_nodes
# Ag = adjacency_matrix(data, num_nodes, scipy_fmt='csr')
# save_npz_matrix('amherst41', Ag)

In [27]:
#save_npz_matrix('Texas')

In [28]:
# g,N = get_graph("amherst41") 
# filename = 'V_amherst41.csv'
# g_sp, Ne_sp, Wsp = generate_spare_graph(g, N, filename)

In [29]:
# data, dataset = get_data_from_dataset('reed98')
#save_weight_list(data, dataset_name=dataset.name, Wsp=Wsp)

In [30]:
def load_data(choice: int):
    if choice < 4:
        if choice == 1:
            data = citegrh.load_cora()
            data_,dataset_ = get_data("Cora")
        elif choice == 2:
            data = citegrh.load_citeseer()
        else:
            data = citegrh.load_pubmed()

        g = data._g
        Ne = g.number_of_edges()
        print(Ne)
        N = g.number_of_nodes()

        # remove self loop
        g = g.remove_self_loop()
        g_sp, Ne_sp = generate_spare_graph(g, N, choice)

        return g, features, labels, mask, test_mask, N, Ne, val_mask

In [31]:
# data = citegrh.load_cora()
# g = data._g
# Ne = g.number_of_edges()
# N = g.number_of_nodes()
# Ag = g.adj_external(scipy_fmt = 'csr')
# print("Saving data now.. ")
# sparse.save_npz(filename,Ag)

In [32]:
#data = gnnbnch.Coauthor('physics')
# g = data[0]
# g = g.remove_self_loop()
# N = data_.num_nodes
# #Ag = g.adj_external(scipy_fmt = 'csr')
# Ag = adjacency_matrix(data_,N)
# print("Saving data now.. ")
# sparse.save_npz(filename,Ag)

## Synthetic Graph

In [33]:
import networkx as nx
import matplotlib.pyplot as plt
import ipynb.fs.full.utils.BarbellGraph as BGraph

def generate_barbell(n_clique=6, n_path = 10):

    clique1 = nx.complete_graph(n_clique)
    clique1_pos = nx.circular_layout(clique1)
    clique2 = nx.complete_graph(n_clique)
    clique2_mapping = {node: node + n_clique for node in clique2}
    nx.relabel_nodes(clique2, clique2_mapping, copy=False) # avoids repeated nodes
    x_diff, y_diff = 4, -1
    clique2_pos = {node: clique1_pos[node-n_clique] + (x_diff, y_diff) for node in clique2}
    path = nx.path_graph(n_path)
    path_mapping = {node: node + 2 * n_clique for node in path}
    nx.relabel_nodes(path, path_mapping, copy=False) # avoids repeated nodes
    path_nodes = list(path.nodes)
    path_half1_nodes = path_nodes[:n_path//2]
    path_half2_nodes = path_nodes[n_path//2:]
    path_dist = 0.8
    clique2_entry = n_clique + n_clique // 2
    path_half1_pos = {node: clique1_pos[0] + (path_dist + i * path_dist, 0) for i, node in enumerate(path_half1_nodes)}
    path_half2_pos = {node: clique2_pos[clique2_entry] - (path_dist + i * path_dist, 0) for i, node in enumerate(path_half2_nodes[::-1])}
    path_pos = {**path_half1_pos, **path_half2_pos}
    barbell = nx.Graph()
    barbell.add_edges_from(clique1.edges)
    barbell.add_edges_from(clique2.edges)
    barbell.add_edges_from(path.edges)
    barbell.add_edges_from([(path_half1_nodes[0], 0), (path_half2_nodes[-1], clique2_entry)])
    clique_pos = {**clique1_pos, **clique2_pos}
    barbell_pos = {**clique_pos, **path_pos}
    
    for (u, v) in barbell.edges():
        nx.set_edge_attributes(barbell, {(u, v): {"weight": 1.0}})

#     plt.figure(figsize=(20, 6))
#     nx.draw(barbell, pos=barbell_pos, with_labels=True, node_size=1000, alpha=0.8, font_size=16)
    
    return barbell, barbell_pos

#barbell, barbell_pos=generate_barbell(6,2)

In [34]:
import networkx as nx
import matplotlib.pyplot as plt

def adjacency_matrix_(edge_index, num_nodes, scipy_fmt='csr'):
    #adj_matrix = torch.zeros((num_nodes, num_nodes), dtype=torch.float)
    edge_index = remove_self_loops(edge_index)[0]
    adj_mat = torch.zeros((num_nodes,num_nodes))
    edges = edge_index.t()
    adj_mat[edges[:,0], edges[:,1]] = 1
    adj_mat[edges[:,1], edges[:,0]] = 1
    #adj_mat[17,14] = 1
    print(adj_mat)
    if scipy_fmt == 'csr':
        adj_matrix_csr = csr_matrix(adj_mat)
        return adj_matrix_csr
    else:
        raise ValueError(f"Unsupported format: {scipy_fmt}")

def create_edge_index(graph):
    edges = list(graph.edges())
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index


def generate_barbell_(n_clique=6):
    # Create the first clique
    clique1 = nx.complete_graph(n_clique)
    clique1_pos = nx.circular_layout(clique1)

    # Create the second clique
    clique2 = nx.complete_graph(n_clique)
    clique2_mapping = {node: node + n_clique for node in clique2}
    nx.relabel_nodes(clique2, clique2_mapping, copy=False)
    x_diff, y_diff = 4, -1
    clique2_pos = {node: clique1_pos[node - n_clique] + (x_diff, y_diff) for node in clique2}

    # Create the third clique
    clique3 = nx.complete_graph(n_clique)
    clique3_mapping = {node: node + 2 * n_clique for node in clique3}
    nx.relabel_nodes(clique3, clique3_mapping, copy=False)
    x_diff_3, y_diff_3 = 8, 0
    clique3_pos = {node: clique1_pos[node - 2 * n_clique] + (x_diff_3, y_diff_3) for node in clique3}

    # Combine all parts into one graph
    barbell = nx.Graph()
    barbell.add_edges_from(clique1.edges)
    barbell.add_edges_from(clique2.edges)
    barbell.add_edges_from(clique3.edges)

    barbell.add_edge(5, 9)
    barbell.add_edge(11,16)
    barbell.add_edge(17,1)

    # Combine all positions
    clique_pos = {**clique1_pos, **clique2_pos, **clique3_pos}
    barbell_pos = clique_pos

    # Set edge attributes
    for (u, v) in barbell.edges():
        nx.set_edge_attributes(barbell, {(u, v): {"weight": 1.0}})

    return barbell, barbell_pos

def draw_graph(G, pos, node_labels=None, weight=None, cosine_weight=None, layout=None):
    plt.figure(figsize=(20, 6))
    
    if layout == 'spring':        
        pos = nx.spring_layout(G)
    elif layout == 'circular':        
        pos = nx.circular_layout(G)
    
    if weight is None:
        if nx.is_weighted(G):
            edge_labels = dict([((u, v,), f"{d['weight']:.2f}") for u, v, d in G.edges(data=True)])
        else:
            edge_labels = dict([((u, v,), f"{1}") for u, v in G.edges()])
    else:
        if cosine_weight is not None:
            edge_labels = dict([((u, v,), f"{weight[u][v]:.2f}, {cosine_weight[u][v]:.2f}") for u, v, d in G.edges(data=True) if weight[u][v] != 0])
        else:
            edge_labels = dict([((u, v,), f"{weight[u][v]:.2f}") for u, v, d in G.edges(data=True) if weight[u][v] != 0])
    
    if node_labels is not None:
        n_clusters = len(np.unique(node_labels))
        colors = plt.cm.rainbow(np.linspace(0, 1, n_clusters))  # Generate colors for clusters
        color_map = [colors[label] for label in node_labels]  # Assign colors based on node labels
        nx.draw(G, pos, with_labels=True, node_color=color_map, cmap=plt.cm.rainbow, alpha=1, node_size=1000, font_size=16)
    else:
        nx.draw(G, pos, with_labels=True, alpha=1, node_size=1000, font_size=16)
    
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
    plt.show()

# # Generate and draw the updated barbell graph
# barbell, barbell_pos = generate_barbell_(6)
# draw_graph(barbell, barbell_pos)

In [35]:
# edge_index = create_edge_index(barbell)
# edge_index

In [36]:
#Ag = adjacency_matrix_(edge_index=edge_index,num_nodes=18)
#save_npz_matrix('Synthetic',Ag=Ag)

In [37]:
# feature_dim = 4
# n_clusters = 3
# n_clique = 6
# node_features = np.zeros((barbell.number_of_nodes(), feature_dim))
# node_labels = np.zeros(barbell.number_of_nodes(), dtype=int)
# # Assign features and labels for each cluster

# for i in range(n_clusters):
#     cluster_features = np.random.rand(1, feature_dim)
#     cluster_nodes = range(i * n_clique, (i + 1) * n_clique)
#     node_features[cluster_nodes] = np.repeat(cluster_features, n_clique, axis=0)
#     node_labels[cluster_nodes] = i

In [38]:
# draw_graph(barbell, barbell_pos,node_labels=node_labels)

In [39]:
# adj_mat = adjacency_matrix_(edge_index=edge_index,num_nodes=18)

In [40]:
# edge_index = create_edge_index(barbell)
# edge_index

In [41]:
# filename = 'V_Synthetic.csv'
# _,_,weight_mat = generate_spare_graph(adj_mat, 18, filename)
#draw_graph(barbell, barbell_pos,node_labels=None,weight=weight_list)

In [42]:
# #cosine_similarity_dictionary
def ER_(u,v, L_inv,N):
    x_u = np.zeros((N,))
    x_v = np.zeros((N,)) 
    x_u[u] = 1
    x_v[v] = 1
    d_uv=x_u-x_v
    R_uv=d_uv.dot(L_inv.dot(d_uv)) ## (x_u-x_v)^T*L'*(x_u-x_v)
    return R_uv

def compute_ER_(Adj, L_inv):
    start_nodes, end_nodes, weights = sparse.find(Adj)

    n = np.shape(Adj)[0]
    Reff = sparse.lil_matrix((n,n))
    for orig, end in zip(start_nodes, end_nodes):
        Reff[orig,end] = ER_(orig, end, L_inv, n)  
    return Reff

def EffectiveResistance(Adj):
    #Adj = nx.adjacency_matrix(G)
    L, D  = csgraph.laplacian(Adj, normed=False, return_diag=True)
    #print(np.allclose(L.todense(), np.diag(D)-Adj)) #verify L=D-A 
    L_inv = linalg.pinv(L.todense())
    Reff=compute_ER_(Adj, L_inv)
    return Reff

def ERprob(Adjacency_matrix, adj_t=None, compute='exact'):
    #rowptr_, col_,_ = adj_t.csr()
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(Adjacency_matrix))
    if DEBUG: print("Computing edge resistances: ... ",compute)
    if compute=='exact':
        print("Exact Resistance")
        Re=EffectiveResistance(Adjacency_matrix)
        weight_matrix = np.zeros(Adjacency_matrix.shape)
        Re = np.maximum(0, Re[start_nodes, end_nodes].toarray())
        for i in range(len(start_nodes)):
            weight_matrix[start_nodes[i],end_nodes[i]] = Re[0][i]
            weight_matrix[end_nodes[i],start_nodes[i]] = Re[0][i]
        return weight_matrix
    else:
        print("Computing Local Effective Resistance")
        Re=LocalEffectiveResistance(adj_t, method=None, eps=0.4, lmbda=0.1)
        return Re
    
def calculate_selection_probability_from_ER(adj_t,effective_resistance_algorithm='exact'):
    sparse_adj = csr_matrix(adj_t)
    Re = ERprob(sparse_adj, adj_t, compute=effective_resistance_algorithm)
    return Re

In [43]:
#Re = calculate_selection_probability_from_ER(adj_mat)
#draw_graph(barbell, barbell_pos,node_labels=None,weight=Re)

In [44]:
def normalize_matrix(matrix):
    row_sums = matrix.sum(axis=1)
    norm_matrix = matrix / row_sums[:,np.newaxis]
    return norm_matrix

def get_ER_weight_matrix(edge_index):
    adj_mat = adjacency_matrix_(edge_index=edge_index,num_nodes=18)
    filename = 'V_Synthetic.csv'
    _,_,weight_mat,ER = generate_spare_graph(adj_mat, 18, filename)
    weight_list = np.zeros((18,18))
    w_matrix = weight_mat.toarray()
    for ind,(u,v) in enumerate(zip(edge_index[0],edge_index[1])):
        print(u,v,w_matrix)
        weight_list[u][v] = w_matrix[u][v]
    return weight_list,ER

def Custom_CosineSimilarity_Sampling_(x, edge_index):
    cosine_similarity_dict = {}
    for node in range(18):
        subset_index = torch.where(edge_index[1] == node)
        indices_list = subset_index[0].tolist()
        subset_nodes = edge_index[0, indices_list]
        neighbor_nodes = subset_nodes.tolist()
        key_vector = x[node]
        value_vectors = x[neighbor_nodes]
        similarities = calculate_cosine_similarity(key_vector, value_vectors)
        #similarities = similarities + 0.01
        #normalized_weights = normalize_weights(similarities)
        cosine_similarity_dict[node] = {'neighbors': neighbor_nodes, 'similarities': similarities}
    return cosine_similarity_dict

def calculate_cosine_weight_(edge_index):
    reverse_edges = edge_index.flip(0)
    undirected_edge_index = torch.cat([edge_index, reverse_edges], dim=1)
    undirected_edge_index = torch.unique(undirected_edge_index, dim=1)
    cosine_dict = Custom_CosineSimilarity_Sampling_(node_features, undirected_edge_index)
    cosine_weight  = np.zeros((18,18))
    for i in range(len(edge_index[0])):
        u,v = edge_index[0][i].item(), edge_index[1][i].item()
        target_index = cosine_dict[u]['neighbors'].index(v)
        edge_weight = cosine_dict[u]['similarities'][target_index]
        cosine_weight[u][v] = edge_weight
        cosine_weight[v][u] = edge_weight
    return cosine_weight

In [45]:
# cosine_weight_ = calculate_cosine_weight_(edge_index)
# ER_weight_list, ER_weight = get_ER_weight_matrix(edge_index)

In [46]:
# draw_graph(barbell, barbell_pos,node_labels=node_labels,weight=er_norm,cosine_weight=cos_norm)

In [47]:
# # Parameters
# feature_dim = 4
# n_clusters = 3
# n_clique = 6
# n_nodes = n_clusters * n_clique
# noise_level = 0.2  # Adjust the noise level to control similarity within clusters

# # Initialize node features and labels
# node_features = np.zeros((barbell.number_of_nodes(), feature_dim))
# node_labels = np.zeros(barbell.number_of_nodes(), dtype=int)

# # Assign features and labels for each cluster
# for i in range(n_clusters):
#     base_feature = np.random.rand(1, feature_dim)
#     cluster_nodes = range(i * n_clique, (i + 1) * n_clique)
#     noise = noise_level * np.random.rand(n_clique, feature_dim)
#     node_features[cluster_nodes] = base_feature + noise
#     node_labels[cluster_nodes] = i

# # Verify the generated features and labels
# print("Node Features:\n", node_features)
# print("Node Labels:\n", node_labels)

In [41]:
#draw_graph(barbell, barbell_pos,node_labels=node_labels,weight=ER_weight_norm,cosine_weight=cos_weight)